# Phase 7: Test the models on your own leaf photos

Your 12 models have only ever seen PlantVillage photos: single leaves, evenly lit, on a plain background. This notebook shows them **your** photos, taken in real sun and shade, and checks three things:

1. **Does the answer change when only the light changes?** Each leaf is photographed in sun, shade and indoors, so any change in the model's answer is caused by the light. This needs no label, so it works for **any** plant, and it's the most direct real-world test of your lighting hypothesis.
2. **Is the model honest about plants it has never seen?** On guava or mango leaves every answer is wrong by definition, because the models only know 15 tomato, potato and pepper classes. A trustworthy model should at least be *unsure*.
3. **Is it right?** Only measurable for tomato, potato and pepper leaves, plus chili as a close relative of bell pepper.

It doesn't train anything, so use a **CPU** runtime (Runtime → Change runtime type → CPU) to save your GPU time.

## Before you run it: take and upload the photos

**Which plants**
- **Tomato, potato or bell pepper:** the plants the models were trained on. These are the only ones where "is it right?" can be checked properly.
- **Green chili:** a close relative of bell pepper (both are *Capsicum*). A healthy chili leaf is checked against "pepper healthy", the closest answer the models have.
- **Any other plant** (guava, mango, lemon, Malabar spinach…): the models will always give one of their 15 disease names, which is wrong by definition. These photos still count for tests 1 and 2 above.

**How to photograph**
- **One leaf per photo, in the middle, filling most of the frame.** The notebook crops a square from the centre. Put a sheet of white paper behind or under the leaf.
- **The same leaf in 3 kinds of light**, one straight after the other, same angle: `sun` (direct sunlight), `shade` (same spot, in shadow), `indoor` (room light). Flash off.

**Folders.** In Google Drive, open `plant_health_capstone`, make a folder called `my_photos`, and inside it one folder per kind of leaf:
- Green chili: `chili_healthy`, or `chili_unsure` if it has spots or yellowing
- Plants the models have never seen: `guava`, `mango`, `lemon`, `malabar_spinach`, or `other_<plant name>` for anything else, e.g. `other_hibiscus`
- Trained plants, if you can get them: `tomato_healthy`, `potato_healthy`, `pepper_healthy`, `tomato_unsure`, `potato_unsure`, `pepper_unsure`, or, only if you're sure of the disease, one of `tomato_early_blight`, `tomato_late_blight`, `tomato_bacterial_spot`, `tomato_leaf_mold`, `tomato_septoria_leaf_spot`, `tomato_spider_mites`, `tomato_target_spot`, `tomato_yellow_leaf_curl_virus`, `tomato_mosaic_virus`, `potato_early_blight`, `potato_late_blight`, `pepper_bacterial_spot`

**File names.** `<leaf>_<light>`, for example `c01_sun.jpg`, `c01_shade.jpg`, `c01_indoor.jpg`. Photos of the same leaf share the first part. iPhone `.heic` photos are fine.

Example:
```
my_photos/
  chili_healthy/     c01_sun.heic  c01_shade.heic  c01_indoor.heic  c02_sun.heic ...
  guava/             g01_sun.heic  g01_shade.heic  g01_indoor.heic ...
  malabar_spinach/   s01_sun.heic  s01_shade.heic  s01_indoor.heic ...
```

### Step 1: Load the toolboxes and the label list
**Expected output:** "Mounted at /content/drive" and "15 classes".

In [ ]:
!pip install -q pillow-heif
import os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image, ImageOps
from pillow_heif import register_heif_opener
register_heif_opener()  # lets iPhone .heic photos open too
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/plant_health_capstone'
PHOTOS_DIR = f'{PROJECT}/my_photos'
RESULTS_DIR = f'{PROJECT}/results'
OUT_DIR = f'{RESULTS_DIR}/my_photos_test'
IMG_SIZE = 224
os.makedirs(OUT_DIR, exist_ok=True)

results = pd.read_csv(f'{RESULTS_DIR}/results.csv')
split = pd.read_csv(f'{RESULTS_DIR}/data_split.csv')
class_names = split.drop_duplicates('label').sort_values('label')['file'].str.split('/').str[0].tolist()

# folders for the plants the models were trained on
TRAINED = {
    'pepper_bacterial_spot': 'Pepper__bell___Bacterial_spot',
    'pepper_healthy': 'Pepper__bell___healthy',
    'potato_early_blight': 'Potato___Early_blight',
    'potato_late_blight': 'Potato___Late_blight',
    'potato_healthy': 'Potato___healthy',
    'tomato_bacterial_spot': 'Tomato_Bacterial_spot',
    'tomato_early_blight': 'Tomato_Early_blight',
    'tomato_late_blight': 'Tomato_Late_blight',
    'tomato_leaf_mold': 'Tomato_Leaf_Mold',
    'tomato_septoria_leaf_spot': 'Tomato_Septoria_leaf_spot',
    'tomato_spider_mites': 'Tomato_Spider_mites_Two_spotted_spider_mite',
    'tomato_target_spot': 'Tomato__Target_Spot',
    'tomato_yellow_leaf_curl_virus': 'Tomato__Tomato_YellowLeaf__Curl_Virus',
    'tomato_mosaic_virus': 'Tomato__Tomato_mosaic_virus',
    'tomato_healthy': 'Tomato_healthy',
    'tomato_unsure': None, 'potato_unsure': None, 'pepper_unsure': None,
}
CHILI = {'chili_healthy': 'Pepper__bell___healthy', 'chili_unsure': None}
NEVER_SEEN = {'guava', 'mango', 'lemon', 'malabar_spinach'}
GROUPS = ['Tomato/potato/pepper', 'Chili', 'Never seen']
assert all(c in class_names for c in list(TRAINED.values()) + list(CHILI.values()) if c), 'label list does not match the training classes'
SHORT = {v: k for k, v in TRAINED.items() if v}  # readable names for answers

def crop_of(name):
    return name.split('_')[0].lower()  # 'tomato', 'potato' or 'pepper'

def folder_info(key):
    # returns (correct class or None, correct plant or None, group), or None if the folder isn't recognised
    if key in TRAINED:
        return TRAINED[key], crop_of(key), 'Tomato/potato/pepper'
    if key in CHILI:
        return CHILI[key], 'pepper', 'Chili'
    if key in NEVER_SEEN or key.startswith('other_'):
        return None, None, 'Never seen'
    return None

print(len(class_names), 'classes')

### Step 2: Find your photos
Reads every photo in `my_photos`: the kind of leaf from the folder name, and the leaf and light from the file name.

**Expected output:** a count of photos and leaves, and a table of photos per folder and light. If it warns about a folder or file, fix the name in Drive and run this cell again.

In [ ]:
rows, problems = [], []
if not os.path.isdir(PHOTOS_DIR):
    raise FileNotFoundError(f'No folder at {PHOTOS_DIR}. Make it in Drive and add your photos first (see the guide above).')
for folder in sorted(pathlib.Path(PHOTOS_DIR).iterdir()):
    if not folder.is_dir():
        continue
    key = folder.name.strip().lower().replace(' ', '_')
    info = folder_info(key)
    if info is None:
        problems.append(f'Folder "{folder.name}" is not one of the names in the guide, so it was skipped.')
        continue
    true_class, true_crop, group = info
    for p in sorted(folder.iterdir()):
        if p.suffix.lower() not in ('.jpg', '.jpeg', '.png', '.heic'):
            continue
        stem = p.stem.strip().lower()
        if '_' in stem:
            leaf, lighting = stem.rsplit('_', 1)
        else:
            leaf, lighting = stem, 'unnamed'
            problems.append(f'"{folder.name}/{p.name}" has no _sun / _shade / _indoor ending, so its light is unknown.')
        rows.append({'photo': f'{folder.name}/{p.name}', 'path': str(p), 'folder': key, 'group': group,
                     'leaf': f'{key}/{leaf}', 'lighting': lighting,
                     'true_class': true_class, 'true_crop': true_crop})

for message in problems:
    print('WARNING:', message)
photos = pd.DataFrame(rows)
if photos.empty:
    raise ValueError('No photos found in my_photos. Check the folder names against the guide above.')
print(len(photos), 'photos of', photos.leaf.nunique(), 'leaves')
pd.crosstab(photos.folder, photos.lighting, margins=True, margins_name='total')

### Step 3: Prepare the photos the same way as the training photos
Each photo is turned the right way up, cropped to a square from the centre, and shrunk to 224 × 224 pixels, the same size the models trained on.

**Expected output:** a grid of your photos as the models will see them. If a leaf got cut off by the square crop, retake it with the leaf more central.

In [ ]:
def load_photo(path):
    img = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
    w, h = img.size
    side = min(w, h)
    left, top = (w - side) // 2, (h - side) // 2
    img = img.crop((left, top, left + side, top + side)).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    return np.asarray(img, dtype=np.uint8)

def convert_colour_space(images, colour_space):  # identical to the training notebook
    images = tf.clip_by_value(tf.cast(images, tf.float32), 0.0, 255.0)
    if colour_space == 'hsv':
        images = tf.image.rgb_to_hsv(images / 255.0) * 255.0
    return images

images = np.stack([load_photo(p) for p in photos.path])

cols = 6
grid_rows = int(np.ceil(len(photos) / cols))
plt.figure(figsize=(2.2 * cols, 2.4 * grid_rows))
for i in range(len(photos)):
    plt.subplot(grid_rows, cols, i + 1)
    plt.imshow(images[i])
    plt.title(photos.photo.iloc[i], fontsize=7)
    plt.axis('off')
plt.tight_layout()
plt.show()

### Step 4: Run all 12 models on your photos
Loads each saved model from Drive and records what it predicts for every photo, and how confident it is. This takes a few minutes on CPU.

**Expected output:** one "done" line per model.

In [ ]:
records, all_probs = [], {}
for _, r in results.iterrows():
    name = f'{r.arch}_{r.colour_space}_seed{r.seed}'
    model = tf.keras.models.load_model(f'{RESULTS_DIR}/{name}/model.keras', compile=False)
    probs = tf.nn.softmax(model(convert_colour_space(images, r.colour_space), training=False)).numpy()
    all_probs[name] = probs
    for i, ph in enumerate(photos.itertuples()):
        predicted = class_names[int(np.argmax(probs[i]))]
        records.append({'run': name, 'arch': r.arch, 'colour_space': r.colour_space, 'seed': r.seed,
                        'photo': ph.photo, 'group': ph.group, 'leaf': ph.leaf, 'lighting': ph.lighting,
                        'true_class': ph.true_class, 'true_crop': ph.true_crop, 'predicted': predicted,
                        'confidence': float(probs[i].max()),
                        'correct': float(predicted == ph.true_class) if isinstance(ph.true_class, str) else np.nan,
                        'crop_correct': float(crop_of(predicted) == ph.true_crop) if isinstance(ph.true_crop, str) else np.nan})
    print(f'{name}: done')
    del model
    tf.keras.backend.clear_session()

preds = pd.DataFrame(records)
preds.to_csv(f'{OUT_DIR}/predictions_all_runs.csv', index=False)

### Step 5: The results
Each score is the average over a model's 3 runs (± the spread between runs).

**Table A: overall**
- **Same answer in every light:** % of leaves where the model gave the same answer in sun, shade and indoors. Higher = more robust to lighting.
- **Lighting shift:** how much the model's answer changes between photos of the **same leaf** in different light, from 0 (identical) to 1 (completely different). Lower = more robust to lighting. This is finer-grained than "same answer", because it also notices when the model's confidence changes.
- **Leaf shift:** the same measure, between **different leaves** in the same light. It's the yardstick for lighting shift: a model should react much more to a different leaf than to different light. **If leaf shift is also close to 0, the model gives nearly the same answer to everything, and its lighting scores mean nothing.**
- **Confidence on never-seen plants:** how sure the model was about guava, mango and so on, where every answer is wrong. Lower = more honest.
- **Correct** and **Right plant:** only for tomato, potato, pepper and chili photos. "Right plant" means it at least named the right plant (pepper for chili).

**Table B** splits the same scores by kind of plant. **Table C** shows "correct" and "right plant" by light, for the plants where those can be checked.

In [ ]:
CONDITIONS = [('mobilenet', 'rgb'), ('mobilenet', 'hsv'), ('scratch', 'rgb'), ('scratch', 'hsv')]
LABELS = {('mobilenet', 'rgb'): 'MobileNet RGB', ('mobilenet', 'hsv'): 'MobileNet HSV',
          ('scratch', 'rgb'): 'Scratch RGB', ('scratch', 'hsv'): 'Scratch HSV'}

def same_answer_rate(d):
    multi = [g for _, g in d.groupby('leaf') if g.lighting.nunique() > 1]
    return np.mean([g.predicted.nunique() == 1 for g in multi]) if multi else np.nan

def answer_change(p, q):
    return 0.5 * np.abs(p - q).sum()  # 0 = identical answers, 1 = completely different

leaf_ids, lights, n = photos.leaf.values, photos.lighting.values, len(photos)
same_leaf_pairs = [(i, j) for i in range(n) for j in range(i + 1, n)
                   if leaf_ids[i] == leaf_ids[j] and lights[i] != lights[j]]
same_light_pairs = [(i, j) for i in range(n) for j in range(i + 1, n)
                    if leaf_ids[i] != leaf_ids[j] and lights[i] == lights[j] and lights[i] != 'unnamed']

def mean_shift(probs, pairs):
    return np.mean([answer_change(probs[i], probs[j]) for i, j in pairs]) if pairs else np.nan

per_run = []
for (arch, colour, seed), d in preds.groupby(['arch', 'colour_space', 'seed']):
    probs = all_probs[f'{arch}_{colour}_seed{seed}']
    unseen = d[d.group == 'Never seen']
    per_run.append({'arch': arch, 'colour_space': colour, 'seed': seed,
                    'Same answer in every light': same_answer_rate(d),
                    'Lighting shift': mean_shift(probs, same_leaf_pairs),
                    'Leaf shift': mean_shift(probs, same_light_pairs),
                    'Confidence on never-seen plants': unseen.confidence.mean() if len(unseen) else np.nan,
                    'Correct': d.correct.mean(), 'Right plant': d.crop_correct.mean()})
per_run = pd.DataFrame(per_run)

def fmt(values, percent=True):
    values = pd.Series(values).dropna()
    if values.empty:
        return 'n/a'
    if percent:
        return f'{100 * values.mean():.0f}% ± {100 * values.std():.0f}' if len(values) > 1 else f'{100 * values.mean():.0f}%'
    return f'{values.mean():.2f} ± {values.std():.2f}' if len(values) > 1 else f'{values.mean():.2f}'

table_a = []
for c in CONDITIONS:
    sub = per_run[(per_run.arch == c[0]) & (per_run.colour_space == c[1])]
    table_a.append({'Model': LABELS[c],
                    'Same answer in every light': fmt(sub['Same answer in every light']),
                    'Lighting shift': fmt(sub['Lighting shift'], percent=False),
                    'Leaf shift': fmt(sub['Leaf shift'], percent=False),
                    'Confidence on never-seen plants': fmt(sub['Confidence on never-seen plants']),
                    'Correct': fmt(sub['Correct']), 'Right plant': fmt(sub['Right plant'])})
table_a = pd.DataFrame(table_a)
table_a.to_csv(f'{OUT_DIR}/table_a_overall.csv', index=False)

table_b = []
for c in CONDITIONS:
    for g in GROUPS:
        d = preds[(preds.arch == c[0]) & (preds.colour_space == c[1]) & (preds.group == g)]
        if d.empty:
            continue
        by_seed = list(d.groupby('seed'))
        table_b.append({'Model': LABELS[c], 'Plants': g, 'Photos': d.photo.nunique(),
                        'Same answer in every light': fmt([same_answer_rate(s) for _, s in by_seed]),
                        'Confidence': fmt([s.confidence.mean() for _, s in by_seed]),
                        'Correct': fmt([s.correct.mean() for _, s in by_seed]),
                        'Right plant': fmt([s.crop_correct.mean() for _, s in by_seed])})
table_b = pd.DataFrame(table_b)
table_b.to_csv(f'{OUT_DIR}/table_b_by_plant.csv', index=False)

print(len(photos), 'photos |', photos.leaf.nunique(), 'leaves |',
      int((photos.groupby('leaf').lighting.nunique() > 1).sum()), 'leaves photographed in more than one light')
print('\nTable A: overall')
display(table_a)
print('\nTable B: by kind of plant')
display(table_b)

checkable = preds[preds.true_crop.notna()]
if checkable.empty:
    print('\nTable C skipped: no tomato, potato, pepper or chili photos, so correctness cannot be checked.')
else:
    table_c = checkable.groupby(['arch', 'colour_space', 'lighting'])[['correct', 'crop_correct']].mean().mul(100).round(0)
    table_c = table_c.rename(columns={'correct': 'Correct %', 'crop_correct': 'Right plant %'}).unstack('lighting')
    table_c = table_c.loc[[c for c in CONDITIONS if c in table_c.index]]
    table_c.index = [LABELS[i] for i in table_c.index]
    table_c.to_csv(f'{OUT_DIR}/table_c_by_light.csv')
    print('\nTable C: correct and right plant, by light')
    display(table_c)

### Step 6: What each model said about each photo
Answers from the first run of each model, with its confidence. Read across a row to compare the four models on the same photo, and down a leaf's photos to see whether its answer changes with the light.

In [ ]:
first = preds[preds.seed == preds.seed.min()].copy()
first['answer'] = first.predicted.map(SHORT) + first.confidence.map(lambda c: f' ({100 * c:.0f}%)')
first['model'] = [LABELS[(a, c)] for a, c in zip(first.arch, first.colour_space)]
per_photo = first.pivot(index='photo', columns='model', values='answer')[[LABELS[c] for c in CONDITIONS]]
per_photo.to_csv(f'{OUT_DIR}/per_photo_answers.csv')
per_photo

### How to read this, and what to say in your paper
- **This is a small test.** With 30–40 photos, describe it as a **case study** that complements the 3,096-photo test set, not as proof.
- **For lighting, the key numbers are "lighting shift" next to "leaf shift".** They compare each leaf only with itself, so they work even for plants the models never learned. Check leaf shift first: if it's near 0, that model answers everything the same way and its lighting scores don't tell you anything.
- **"Confidence on never-seen plants" shows a real limitation.** These models have no "I don't know" option, so they always name one of their 15 classes, often confidently. That's an important point for any real garden tool, and worth a paragraph in your discussion.
- **Chili isn't bell pepper,** only a close relative. Table A combines it with any tomato, potato or pepper photos, so when you report chili results, use its own rows in Table B.

Everything is saved in `results/my_photos_test/`.